In [3]:
# CELL 1: DEPENDENCY INSTALLATION
!pip install -q transformers datasets accelerate soundfile librosa jiwer evaluate torchaudio

In [4]:
# CELL 2: DATASET EXTRACTION AND PATH RESOLUTION
import os
import zipfile
import pandas as pd
import unicodedata
from sklearn.model_selection import train_test_split

# Path to the uploaded zip archive
zip_path = '/content/drive/MyDrive/corpus/chunks.zip'
extract_path = '/content/dataset_extracted'

# Unzip dataset
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# Path to metadata CSV
csv_path = os.path.join(extract_path, 'chunks/Acts1,2_aligned_metadata.csv')

# Load metadata
df = pd.read_csv(csv_path)

# Preprocess and normalize transcriptions (NFC Format)
def clean_and_normalize(text):
    if not isinstance(text, str):
        return ""
    # Normalize to Unicode NFC format to keep combined diacritics correct
    text_normalized = unicodedata.normalize('NFC', text)
    # Keep only standard alphabetic characters, Nuer diacritics, spaces, and standard slash '/' (for negations)
    # We do NOT strip '/' from the beginning of words
    return text_normalized.strip()

df['transcription_clean'] = df['transcription'].apply(clean_and_normalize)

# Construct the full path
# Since folder contents were extracted under extract_path, the path should resolve there
df['audio_path'] = df.apply(lambda row: os.path.join(extract_path, 'chunks', row['folder'], row['file_name']), axis=1)

# Check if file paths exist
df['file_exists'] = df['audio_path'].apply(os.path.exists)
df = df[df['file_exists'] == True].reset_index(drop=True)
print(f"Total valid audio samples found: {len(df)}")

# Split dataset into 90% training and 10% validation
train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Train size: {len(train_df)} | Eval size: {len(eval_df)}")

Total valid audio samples found: 107
Train size: 96 | Eval size: 11


In [5]:
# CELL 3: CUSTOM NUER VOCABULARY AND TOKENIZER CREATION
import json
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

# Extract unique vocabulary characters from normalized transcriptions
def extract_all_chars(df):
    all_text = " ".join(df['transcription_clean'].tolist())
    vocab = list(set(all_text))
    return vocab

# Build custom Nuer vocabulary dictionary
vocab_list = extract_all_chars(train_df)

# Create vocabulary map with indices
vocab_dict = {v: i for i, v in enumerate(sorted(vocab_list))}

# Replace space with '|' representation
if ' ' in vocab_dict:
    vocab_dict['|'] = vocab_dict.pop(' ')

# Ensure CTC and special tokens are included
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

# Save vocabulary JSON file
with open('vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

print("Nuer Vocabulary Size:", len(vocab_dict))
print("Vocabulary Map:", vocab_dict)

# Instantiate CTC Tokenizer
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

# Instantiate Feature Extractor
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

# Combine into an unified processor
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

Nuer Vocabulary Size: 33
Vocabulary Map: {'/': 1, 'a': 2, 'b': 3, 'c': 4, 'd': 5, 'e': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'r': 17, 't': 18, 'u': 19, 'w': 20, 'y': 21, 'ä': 22, 'ë': 23, 'ö': 24, 'ŋ': 25, 'ɔ': 26, 'ɛ': 27, 'ɣ': 28, '̈': 29, '̱': 30, '|': 0, '[UNK]': 31, '[PAD]': 32}


In [6]:
# CELL 4: HUGGING FACE DATASET CONVERSION & PREPROCESSING
from datasets import Dataset, Audio

# Convert pandas dataframes to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[['audio_path', 'transcription_clean']])
eval_dataset = Dataset.from_pandas(eval_df[['audio_path', 'transcription_clean']])

# Cast file paths to Hugging Face Audio Feature for automatic loading and resample to 16kHz
train_dataset = train_dataset.cast_column("audio_path", Audio(sampling_rate=16000))
eval_dataset = eval_dataset.cast_column("audio_path", Audio(sampling_rate=16000))

# Preprocessing mapping function
def prepare_dataset(batch):
    audio = batch["audio_path"]

    # Process raw audio array
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    # Process target labels (character token IDs) without as_target_processor()
    batch["labels"] = processor(text=batch["transcription_clean"]).input_ids
    return batch

# Map datasets
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(prepare_dataset, remove_columns=eval_dataset.column_names)

print("Sample processed inputs:", train_dataset[0].keys())

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Sample processed inputs: dict_keys(['input_values', 'input_length', 'labels'])


In [7]:
# CELL 5: CTC DATA COLLATOR CLASS
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Separate inputs and labels since they require different padding strategies
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Pad audio inputs
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Pad label IDs using the tokenizer explicitly
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Replace padding labels with -100 to ignore loss calculations
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [8]:
# CELL 6: MMS 1B MODEL & ADAPTER INITIALIZATION
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Config

# Detect available accelerator device (Make sure you switched Colab to a GPU!)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: You are running on CPU. Training a 1B model will be extremely slow. Please switch to a GPU runtime.")

# 1. Load the configuration of facebook/mms-1b
config = Wav2Vec2Config.from_pretrained("facebook/mms-1b")

# 2. Update configuration parameters explicitly to enable the target adapter mechanism and apply dropouts
config.update({
    "adapter_attn_dim": 16, # Default dimension used by MMS language adapters
    "vocab_size": len(processor.tokenizer),
    "pad_token_id": processor.tokenizer.pad_token_id,
    "bos_token_id": processor.tokenizer.bos_token_id if hasattr(processor.tokenizer, 'bos_token_id') else 1,
    "eos_token_id": processor.tokenizer.eos_token_id if hasattr(processor.tokenizer, 'eos_token_id') else 2,
    "ctc_loss_reduction": "mean",
    "attention_dropout": 0.0,
    "hidden_dropout": 0.0,
    "feat_proj_dropout": 0.0,
    "mask_time_prob": 0.0,
    "layerdrop": 0.0
})

# 3. Instantiate model with this customized configuration.
# Setting adapter_attn_dim in config automatically instantiates the adapter layers inside the model layers!
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/mms-1b",
    config=config,
    ignore_mismatched_sizes=True
)

# 4. Freeze the base backbone
model.freeze_base_model()

# 5. Set Nuer 'nus' as the active adapter language
# Since we configured the model with 'adapter_attn_dim', the adapter weights exist and are now trainable
try:
    # Attempt to load target language weights if they exist natively
    model.load_adapter("nus")
    print("Successfully loaded pre-trained adapter weights for 'nus'")
except Exception as e:
    print(f"Using newly initialized target adapter layers for 'nus'.")

# 6. Ensure only the adapter layers and output projection head (lm_head) are trainable
for name, param in model.named_parameters():
    if "adapter" in name or "lm_head" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Explicitly move model weights to selected hardware accelerator device
model = model.to(device)
print("Model adapter initialization complete and moved to target device.")

Using device: cpu


Loading weights:   0%|          | 0/805 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b
Key                                                            | Status     | 
---------------------------------------------------------------+------------+-
quantizer.weight_proj.weight                                   | UNEXPECTED | 
quantizer.codevectors                                          | UNEXPECTED | 
wav2vec2.masked_spec_embed                                     | UNEXPECTED | 
project_q.bias                                                 | UNEXPECTED | 
project_q.weight                                               | UNEXPECTED | 
project_hid.bias                                               | UNEXPECTED | 
project_hid.weight                                             | UNEXPECTED | 
quantizer.weight_proj.bias                                     | UNEXPECTED | 
wav2vec2.encoder.layers.{0...47}.adapter_layer.linear_2.weight | MISSING    | 
wav2vec2.encoder.layers.{0...47}.adapter_layer.linear_1.weight | MI

Using newly initialized target adapter layers for 'nus'.
Model adapter initialization complete and moved to target device.


In [ ]:
# CELL 7: METRICS, TRAINING ARGUMENTS & EXECUTION
import evaluate
from transformers import TrainingArguments, Trainer
import numpy as np
import torch

# Load Word Error Rate metric
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions and labels back to text
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# Set training hyper-parameters
use_fp16 = torch.cuda.is_available() # Automatically turn off fp16 if running on CPU

training_args = TrainingArguments(
    output_dir="./mms-1b-nuer-adapter",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    weight_decay=0.005,
    num_train_epochs=10,
    fp16=use_fp16,            # Set according to system compatibility
    save_total_limit=2,
    logging_steps=10,
    metric_for_best_model="wer",
    greater_is_better=False,
    load_best_model_at_end=True,
    dataloader_num_workers=2,
    report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
)

# Train the Adapter modules
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
